# Monte Carlo Simulations Using SIMPA, modelling tissues: muscle, fat, and phantoms: co-polymer-in-oil, PDMS & Agar 


In [ ]:
from simpa import Tags
import simpa as sp
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# FIXME temporary workaround for newest Intel architectures
import os
os.environ["KMP_DUPLICATE_LIB_OK"] = "TRUE"

In [ ]:
# TODO: Please make sure that a valid path_config.env file is located in your home directory, or that you
#  point to the correct file in the PathManager().
path_manager = sp.PathManager()

VOLUME_TRANSDUCER_DIM_IN_MM = 20 # x 20
VOLUME_PLANAR_DIM_IN_MM = 20 # y 20
VOLUME_HEIGHT_IN_MM = 20 # z 20
SPACING = 0.10
RANDOM_SEED = 471
VOLUME_NAME = "test"
SAVE_REFLECTANCE = True
SAVE_PHOTON_DIRECTION = True

# If VISUALIZE is set to True, the simulation result will be plotted
VISUALIZE = True

# Define SWIR tissue models

1. define moleucles using Molecule: input mua, mus & g data

2. feed into all molecules that make up the tissue and their fractions into MolecularCompositionGenerator

starting with water, fat, background soft tissue and muscle:


In [ ]:
# load g, mua and mus spectra

#------------------------------------------------------------------------------------------------------------------------------------------------------
# PATH to simulation spectra
#PATH = '/Users/melissa/Library/CloudStorage/OneDrive-UniversityofCambridge/PhD/Lab work/Biomolecule characterisation/Data_and_Code/Data/Simulation_Spectra'
PATH = r"D:\OneDrive - University of Cambridge\PhD\Lab work\Biomolecule characterisation\Data_and_Code\Data\Simulation_Spectra/"
#------------------------------------------------------------------------------------------------------------------------------------------------------
# function to remove NaNs from spectra
def dropna(df):
    df_filtered = df.replace([np.inf, -np.inf], np.nan).dropna()
    return df_filtered
#------------------------------------------------------------------------------------------------------------------------------------------------------
## PHANTOMS / MUA & MUSP [cm-1] ##

# PDMS phantom
PDMS = pd.read_csv(PATH+"PDMS_mua_musp.txt", sep='\t')

# co-polymer-in-oil phantom
copolymer= pd.read_csv(PATH+"copolymer_mua_musp.txt", sep='\t')

# agar phantom 
agar = pd.read_csv(PATH+"agar_mua_musp.txt", sep='\t')
#------------------------------------------------------------------------------------------------------------------------------------------------------
## TISSUE COMPONENTS ABSORPTION / MUA [cm-1] ##
    
# fat - van Veen + MJW
fat_mua = pd.read_csv(PATH+"fat_mua.txt", sep='\t') # pig fat (van Veen) 450-1000nm, lard (MJW) 1000-1600nm
fat_mua = dropna(fat_mua)

# oxy blood - MJW
oxy_blood_mua = pd.read_csv(PATH+"oxy_blood_mua.txt", sep='\t')
oxy_blood_mua = dropna(oxy_blood_mua)

# deoxy blood - MJW
deoxy_blood_mua = pd.read_csv(PATH+"deoxy_blood_mua.txt", sep='\t')
deoxy_blood_mua = dropna(deoxy_blood_mua)

# water - Segelstein
water_mua = pd.read_csv(PATH+"water_mua.txt", sep='\t')

#------------------------------------------------------------------------------------------------------------------------------------------------------
## TISSUE COMPONENTS REDUCED SCATTERING / MUSP [cm-1] ##

# fat
fat_musp = pd.read_csv(PATH+"fat_musp.txt", sep='\t')

# blood
blood_musp = pd.read_csv(PATH+"blood_musp.txt", sep='\t')

# muscle
muscle_musp = pd.read_csv(PATH+"muscle_musp.txt", sep='\t')

#------------------------------------------------------------------------------------------------------------------------------------------------------
# insert citation for reference spectra

In [ ]:
# Plot the spectra

plt.rc('font', size=12) 
fig, ax = plt.subplots(1, 2, figsize=(9, 3.5))

# absorption
ax[0].plot(oxy_blood_mua["nm"], oxy_blood_mua["mua[cm-1]"], label="Oxygenated blood", color="orangered")
ax[0].plot(deoxy_blood_mua["nm"], deoxy_blood_mua["mua[cm-1]"], label="Deoxygenated blood", color="royalblue")
ax[0].plot(water_mua["nm"], water_mua["mua[cm-1]"], label="Water", color="k", ls='--')
filter_fat_vis = fat_mua[fat_mua["nm"] < 1000]
ax[0].plot(filter_fat_vis["nm"], filter_fat_vis["mua[cm-1]"], label=None, color="forestgreen", ls='--')
filter_fat_SWIR = fat_mua[fat_mua["nm"] > 1000]
ax[0].plot(filter_fat_SWIR["nm"], filter_fat_SWIR["mua[cm-1]"], label="Fat", color="forestgreen")

# scattering

# define colours
c_blood = (161/255,52/255,55/255)
c_fat = (255/255,215/255,151/255)
c_muscle = (196/255,120/255,106/255)

ax[1].plot(blood_musp["nm"], blood_musp["musp[cm-1]"], label="Blood", color=c_blood, ls='--')
ax[1].plot(fat_musp["nm"], fat_musp["musp[cm-1]"], label="Fatty tissue", color=c_fat, ls='--')
ax[1].plot(muscle_musp["nm"], muscle_musp["musp[cm-1]"], label="Muscle", color=c_muscle, ls='--')

ax[0].set_yscale("log")
ax[1].set_yscale("log")

ax[0].set_xlim(450, 1600)
ax[1].set_xlim(450, 1600)

ax[0].set_ylim(0.001, 10000)
ax[1].set_ylim(1, 100)

ax[0].legend(loc=1, fontsize=9, frameon=False)
ax[1].legend(loc=0, fontsize=9, frameon=False)

ax[0].tick_params(axis='both', which='both', direction='in', top=True, right=True, bottom=True)
ax[0].minorticks_on()
ax[1].tick_params(axis='both', which='both', direction='in', top=True, right=True, bottom=True)
ax[1].minorticks_on()

ax[0].set_ylabel('$\mu_a$ (cm$^{-1}$)')
ax[0].set_xlabel('Wavelength (nm)')
ax[1].set_ylabel('$\mu_s^{\prime}$ (cm$^{-1}$)')
ax[1].set_xlabel('Wavelength (nm)')

plt.tight_layout()


In [ ]:
# Plot the phantom spectra

plt.rc('font', size=12) 
fig, ax = plt.subplots(1, 2, figsize=(9, 3.5))

# absorption
ax[0].plot(PDMS["nm"], PDMS["mua[cm-1]"], label="PDMS (nigrosin + TiO2)", color="g")
ax[0].plot(copolymer["nm"], copolymer["mua[cm-1]"], label="co-polymer-in-oil (nigrosin + TiO2)", color="k")
ax[0].plot(agar["nm"], agar["mua[cm-1]"], label="agar (blood + TiO2)", color="r")


# scattering
ax[1].plot(PDMS["nm"], PDMS["musp[cm-1]"], label="PDMS", color="g")
ax[1].plot(copolymer["nm"], copolymer["musp[cm-1]"], label="co-polymer-in-oil", color="k")
ax[1].plot(agar["nm"], agar["musp[cm-1]"], label="agar", color="r")

ax[0].set_yscale("log")
ax[1].set_yscale("log")

ax[0].set_xlim(450, 1600)
ax[1].set_xlim(450, 1600)

ax[0].set_ylim(0.1, 20)
ax[1].set_ylim(1, 20)

ax[0].legend(loc=1, fontsize=9, frameon=False)
ax[1].legend(loc=0, fontsize=9, frameon=False)

ax[0].tick_params(axis='both', which='both', direction='in', top=True, right=True, bottom=True)
ax[0].minorticks_on()
ax[1].tick_params(axis='both', which='both', direction='in', top=True, right=True, bottom=True)
ax[1].minorticks_on()

ax[0].set_ylabel('$\mu_a$ (cm$^{-1}$)')
ax[0].set_xlabel('Wavelength (nm)')
ax[1].set_ylabel('$\mu_s^{\prime}$ (cm$^{-1}$)')
ax[1].set_xlabel('Wavelength (nm)')

plt.tight_layout()


In [ ]:
## ANISOTROPY / G ##

# g values from SIMPA / literature
WATER_G = 1.0
PHANTOM_G = 0.7
STANDARD_G = 0.9  # Average anisotropy of measured values presented in paper
DERMIS_G = 0.715
BLOOD_G = 0.98
COPOLYMER_G = 0.7

# water
water_g_spectrum = sp.Spectrum('water', np.asarray([100, 2000]), np.asarray([WATER_G, WATER_G]))
# blood
blood_g_spectrum = sp.Spectrum('blood', np.asarray([100, 2000]), np.asarray([BLOOD_G, BLOOD_G]))
# muscle scatterer
muscle_scatterer_g_spectrum = sp.Spectrum('muscle scatterer', np.asarray([100, 2000]), np.asarray([STANDARD_G, STANDARD_G]))
# standard scatterer
standard_g_spectrum = sp.Spectrum('standard scatterer', np.asarray([100, 2000]), np.asarray([STANDARD_G, STANDARD_G]))
# custom_water scatterer (see SIMPA documentation)
custom_water_g_spectrum = sp.Spectrum('custom water scatterer', np.asarray([100, 2000]), np.asarray([STANDARD_G-0.005, STANDARD_G-0.005]))

# co-polymer-in-oil phantom
copolymer_g_spectrum = sp.Spectrum('copolymer', np.asarray([100, 2000]), np.asarray([COPOLYMER_G, COPOLYMER_G]))
# agar
agar_g_spectrum = sp.Spectrum('agar', np.asarray([100, 2000]), np.asarray([STANDARD_G, STANDARD_G]))
# PDMS
PDMS_g_spectrum = sp.Spectrum('PDMS', np.asarray([100, 2000]), np.asarray([STANDARD_G, STANDARD_G]))

In [ ]:
# Define spectra loaded above as simpa spectra

#------------------------------------------------------------------------------------------------------------------------------------------------------
## ABSORPTION / MUA [cm-1] ##

# phantoms
# co-polymer-in-oil
copolymer_mua_spectrum = sp.Spectrum('copolymer', copolymer["nm"].values, copolymer["mua[cm-1]"].values)

# agar
agar_mua_spectrum = sp.Spectrum('agar', agar["nm"].values, agar["mua[cm-1]"].values)

# PDMS
PDMS_mua_spectrum = sp.Spectrum('PDMS', PDMS["nm"].values, PDMS["mua[cm-1]"].values)

# biomolecules
# water
water_mua_spectrum = sp.Spectrum('water', water_mua["nm"].values, water_mua["mua[cm-1]"].values)

# HbO2
HbO2_mua_spectrum = sp.Spectrum('HbO2', oxy_blood_mua["nm"].values, oxy_blood_mua["mua[cm-1]"].values)

# Hb
Hb_mua_spectrum = sp.Spectrum('Hb', deoxy_blood_mua["nm"].values, deoxy_blood_mua["mua[cm-1]"].values)

# fat
fat_mua_spectrum = sp.Spectrum('fat', fat_mua["nm"].values, fat_mua["mua[cm-1]"].values)

# muscle scatterer (constant 1e-20 from SIMPA)
muscle_scatterer_mua_spectrum = sp.Spectrum('muscle scatterer', np.asarray([100, 2000]), np.asarray([1e-20, 1e-20]))

# constant absorber (constant 1e-20 from SIMPA)
constant_absorber_mua_spectrum = sp.Spectrum('constant absorber', np.asarray([100, 2000]), np.asarray([1e-20, 1e-20]))

#------------------------------------------------------------------------------------------------------------------------------------------------------
## SCATTERING / MUS [cm-1] ## convert musp to mus for SIMPA

# phantoms
# co-polymer-in-oil
copolymer_mus_spectrum = sp.Spectrum('copolymer', copolymer["nm"].values, copolymer["musp[cm-1]"].values/(1-COPOLYMER_G))

# agar
agar_mus_spectrum = sp.Spectrum('agar', agar["nm"].values, agar["musp[cm-1]"].values/(1-STANDARD_G))

# PDMS
PDMS_mus_spectrum = sp.Spectrum('PDMS', PDMS["nm"].values, PDMS["musp[cm-1]"].values/(1-STANDARD_G))

# biomolecules
# water (WATER_MUS = 1e-10 from SIMPA)
water_mus_spectrum = sp.Spectrum('water', np.asarray([100, 2000]), np.asarray([1e-10, 1e-10]))

# muscle
muscle_mus_spectrum = sp.Spectrum('muscle', muscle_musp["nm"].values, muscle_musp["musp[cm-1]"].values/(1-STANDARD_G))

# blood
blood_mus_spectrum = sp.Spectrum('blood', blood_musp["nm"].values, blood_musp["musp[cm-1]"].values/(1-BLOOD_G))

# fat
fat_mus_spectrum = sp.Spectrum('fat', fat_musp["nm"].values, fat_musp["musp[cm-1]"].values/(1-STANDARD_G))


In [ ]:
## VOLUME FRACTIONS ##
# from OpticalTissueProperties in literature_values.py

WATER_VOLUME_FRACTION_HUMAN_BODY = 0.68
BLOOD_VOLUME_FRACTION_MUSCLE_TISSUE = 0.01  # Value of arterial bvf at t0 in fig 3.
blood_volume_fraction_muscle = 0.01


# Define Molecules

In [ ]:
# Define molecules:
# name
# absorption spectrum
# volume fraction
# anisotropy spectrum
# scattering spectrum

# co-polymer-in-oil
copolymer_molecule = sp.Molecule(name='Copolymer molecule', absorption_spectrum=copolymer_mua_spectrum, volume_fraction=1.0,
                            anisotropy_spectrum=copolymer_g_spectrum, scattering_spectrum=copolymer_mus_spectrum)

# agar
agar_molecule = sp.Molecule(name='agar molecule', absorption_spectrum=agar_mua_spectrum, volume_fraction=1.0,
                            anisotropy_spectrum=agar_g_spectrum, scattering_spectrum=agar_mus_spectrum)

# PDMS
PDMS_molecule = sp.Molecule(name='PDMS molecule', absorption_spectrum=PDMS_mua_spectrum, volume_fraction=1.0,
                            anisotropy_spectrum=PDMS_g_spectrum, scattering_spectrum=PDMS_mus_spectrum)

# water
water_molecule = sp.Molecule(name='Water molecule', absorption_spectrum=water_mua_spectrum, volume_fraction=WATER_VOLUME_FRACTION_HUMAN_BODY,
                            anisotropy_spectrum=water_g_spectrum, scattering_spectrum=water_mus_spectrum)

# water for muscle tissue (contains muscle scattering so muscle tissue contains scattering contributions from the muscle scatterer and here as per SIMPA documentaion: https://simpa.readthedocs.io/en/main/_modules/simpa/utils/libraries/tissue_library.html#TissueLibrary.muscle)
custom_water_molecule = sp.Molecule(name='Custom water molecule', absorption_spectrum=water_mua_spectrum, volume_fraction=WATER_VOLUME_FRACTION_HUMAN_BODY,
                            anisotropy_spectrum=custom_water_g_spectrum, scattering_spectrum=muscle_mus_spectrum)

In [ ]:
# define molecule functions to enable variation of so2 in muscle and sucutaneous fat tissues


# define muscle scatterer molecule
def muscle_scatterer(volume_fraction=1.0):

    return sp.Molecule(
        name="Muscle scatterer molecule",
        absorption_spectrum=muscle_scatterer_mua_spectrum, 
        volume_fraction=volume_fraction,
        scattering_spectrum=muscle_mus_spectrum,
        anisotropy_spectrum=muscle_scatterer_g_spectrum
    )

# define fat scatterer molecule
def fat_scatterer(volume_fraction=1.0):

    return sp.Molecule(
        name="Fat scatterer molecule",
        absorption_spectrum=fat_mua_spectrum, # pig fat mua
        volume_fraction=volume_fraction,
        scattering_spectrum=fat_mus_spectrum,
        anisotropy_spectrum=standard_g_spectrum
    )
    
# define HbO2 molecule
def HbO2(volume_fraction=1.0):

    return sp.Molecule(
        name="HbO2 molecule",
        absorption_spectrum=HbO2_mua_spectrum, 
        volume_fraction=volume_fraction,
        scattering_spectrum=blood_mus_spectrum,
        anisotropy_spectrum=blood_g_spectrum
    )

# define Hb molecule
def Hb(volume_fraction=1.0):

    return sp.Molecule(
        name="Hb molecule",
        absorption_spectrum=Hb_mua_spectrum, 
        volume_fraction=volume_fraction,
        scattering_spectrum=blood_mus_spectrum,
        anisotropy_spectrum=blood_g_spectrum
    )


# muscle tissue
def muscle_fn(oxygenation, blood_volume_fraction):
        """
        Create a molecular composition mimicking that of muscle
        :param oxygenation - oxygenation level in the tissue
        :param blood_volume_fraction - tissue specific BVF
        :return: a settings dictionary containing all min and max parameters fitting for muscle tissue.
        """
        fraction_oxy = blood_volume_fraction * oxygenation 
        fraction_deoxy = blood_volume_fraction * (1 - oxygenation)
    
        # generate the tissue dictionary
        return (sp.MolecularCompositionGenerator()
                .append(muscle_scatterer(1 - fraction_oxy - fraction_deoxy - WATER_VOLUME_FRACTION_HUMAN_BODY))
                .append(custom_water_molecule)
                .append(HbO2(fraction_oxy))
                .append(Hb(fraction_deoxy))
                .get_molecular_composition(sp.SegmentationClasses.MUSCLE))

# fat tissue
def subcutaneous_fat_fn(oxygenation, blood_volume_fraction):
        """
        Create a molecular composition mimicking that of subcutaneous fat
        :param oxygenation - oxygenation level in the tissue        
        :param blood_volume_fraction - tissue specific BVF
        :return: a settings dictionary containing all min and max parameters fitting for fat tissue.
        """
        fraction_oxy = blood_volume_fraction * oxygenation 
        fraction_deoxy = blood_volume_fraction * (1 - oxygenation)
    
        # generate the tissue dictionary
        return (sp.MolecularCompositionGenerator()
                .append(fat_scatterer(1 - fraction_oxy - fraction_deoxy - WATER_VOLUME_FRACTION_HUMAN_BODY))
                .append(water_molecule)
                .append(HbO2(fraction_oxy))
                .append(Hb(fraction_deoxy))
                .get_molecular_composition(sp.SegmentationClasses.FAT))


In [ ]:
# MolecularCompositionGenerator

copolymer_tissue = sp.MolecularCompositionGenerator().append(copolymer_molecule, 'copolymer').get_molecular_composition(sp.SegmentationClasses.GENERIC)

agar_tissue = sp.MolecularCompositionGenerator().append(agar_molecule, 'agar').get_molecular_composition(sp.SegmentationClasses.GENERIC)

PDMS_tissue = sp.MolecularCompositionGenerator().append(PDMS_molecule, 'PDMS').get_molecular_composition(sp.SegmentationClasses.GENERIC)

# oxygenation levels and blood volume fractions for tissues
oxygenation_level = 0.95 # choose 95% for all tissue types
blood_volume_fraction_tissue = 0.01 # 2-5%, https://omlc.org/news/jan98/skinoptics.html

muscle_tissue = muscle_fn(oxygenation_level, blood_volume_fraction_tissue)

subcutaneous_fat_tissue = subcutaneous_fat_fn(oxygenation_level, blood_volume_fraction_tissue) 

blood_tissue = sp.MolecularCompositionGenerator().append(HbO2(oxygenation_level)).append(Hb(1 - oxygenation_level)).get_molecular_composition(sp.SegmentationClasses.BLOOD)


# Library of tissue structures

In [ ]:
# cube containing just copolymer phantom material
def copolymer_tissue_structure():
    """
    Slab of phantom material.
    """
    phantom_dictionary = sp.Settings()
    phantom_dictionary[Tags.PRIORITY] = 1
    phantom_dictionary[Tags.STRUCTURE_START_MM] = [0, 0, 0]
    phantom_dictionary[Tags.STRUCTURE_END_MM] = [0, 0, 100]
    phantom_dictionary[Tags.MOLECULE_COMPOSITION] = copolymer_tissue
    phantom_dictionary[Tags.CONSIDER_PARTIAL_VOLUME] = False
    phantom_dictionary[Tags.ADHERE_TO_DEFORMATION] = False
    phantom_dictionary[Tags.STRUCTURE_TYPE] = Tags.HORIZONTAL_LAYER_STRUCTURE
    
    tissue_dict = sp.Settings()
    tissue_dict["copolymer"] = phantom_dictionary

    return tissue_dict

In [ ]:
# cube containing just agar phantom material
def agar_tissue_structure():
    """
    Slab of phantom material.
    """
    phantom_dictionary = sp.Settings()
    phantom_dictionary[Tags.PRIORITY] = 1
    phantom_dictionary[Tags.STRUCTURE_START_MM] = [0, 0, 0]
    phantom_dictionary[Tags.STRUCTURE_END_MM] = [0, 0, 100]
    phantom_dictionary[Tags.MOLECULE_COMPOSITION] = agar_tissue
    phantom_dictionary[Tags.CONSIDER_PARTIAL_VOLUME] = False
    phantom_dictionary[Tags.ADHERE_TO_DEFORMATION] = False
    phantom_dictionary[Tags.STRUCTURE_TYPE] = Tags.HORIZONTAL_LAYER_STRUCTURE
    
    tissue_dict = sp.Settings()
    tissue_dict["agar"] = phantom_dictionary

    return tissue_dict

In [ ]:
# cube containing just PDMS phantom material
def PDMS_tissue_structure():
    """
    Slab of phantom material.
    """
    phantom_dictionary = sp.Settings()
    phantom_dictionary[Tags.PRIORITY] = 1
    phantom_dictionary[Tags.STRUCTURE_START_MM] = [0, 0, 0]
    phantom_dictionary[Tags.STRUCTURE_END_MM] = [0, 0, 100]
    phantom_dictionary[Tags.MOLECULE_COMPOSITION] = PDMS_tissue
    phantom_dictionary[Tags.CONSIDER_PARTIAL_VOLUME] = False
    phantom_dictionary[Tags.ADHERE_TO_DEFORMATION] = False
    phantom_dictionary[Tags.STRUCTURE_TYPE] = Tags.HORIZONTAL_LAYER_STRUCTURE
    
    tissue_dict = sp.Settings()
    tissue_dict["PDMS"] = phantom_dictionary

    return tissue_dict

In [ ]:
# cube containing just fat
def subcutaneous_fat_tissue_structure():
    """
    Slab of subcutaneous fat.
    """
    subcutaneous_fat_dictionary = sp.Settings()
    subcutaneous_fat_dictionary[Tags.PRIORITY] = 1
    subcutaneous_fat_dictionary[Tags.STRUCTURE_START_MM] = [0, 0, 0]
    subcutaneous_fat_dictionary[Tags.STRUCTURE_END_MM] = [0, 0, 100]
    subcutaneous_fat_dictionary[Tags.MOLECULE_COMPOSITION] = subcutaneous_fat_tissue
    subcutaneous_fat_dictionary[Tags.CONSIDER_PARTIAL_VOLUME] = False
    subcutaneous_fat_dictionary[Tags.ADHERE_TO_DEFORMATION] = False
    subcutaneous_fat_dictionary[Tags.STRUCTURE_TYPE] = Tags.HORIZONTAL_LAYER_STRUCTURE
    
    tissue_dict = sp.Settings()
    tissue_dict["subcutaneous fat"] = subcutaneous_fat_dictionary

    return tissue_dict

In [ ]:
# cube containing just muscle
def muscle_tissue_structure():
    """
    Slab of muscle.
    """
    muscle_dictionary = sp.Settings()
    muscle_dictionary[Tags.PRIORITY] = 1
    muscle_dictionary[Tags.STRUCTURE_START_MM] = [0, 0, 0]
    muscle_dictionary[Tags.STRUCTURE_END_MM] = [0, 0, 100]
    muscle_dictionary[Tags.MOLECULE_COMPOSITION] = muscle_tissue
    muscle_dictionary[Tags.CONSIDER_PARTIAL_VOLUME] = False
    muscle_dictionary[Tags.ADHERE_TO_DEFORMATION] = False
    muscle_dictionary[Tags.STRUCTURE_TYPE] = Tags.HORIZONTAL_LAYER_STRUCTURE
    
    tissue_dict = sp.Settings()
    tissue_dict["muscle"] = muscle_dictionary

    return tissue_dict

# Run loop of simulations

In [ ]:
# sweep sheet illum FP types
def run_simulation(simulation_structure, VOL_NAME):
    
    # Seed the numpy random configuration prior to creating the global_settings file in
    # order to ensure that the same volume
    # is generated with the same random seed every time.

    np.random.seed(RANDOM_SEED)
    
    general_settings = {
        # These parameters set the general properties of the simulated volume
        Tags.RANDOM_SEED: RANDOM_SEED,


        Tags.VOLUME_NAME: VOL_NAME, 
        Tags.SIMULATION_PATH: path_manager.get_hdf5_file_save_path(),
        Tags.SPACING_MM: SPACING,
        Tags.DIM_VOLUME_Z_MM: VOLUME_HEIGHT_IN_MM,
        Tags.DIM_VOLUME_X_MM: VOLUME_TRANSDUCER_DIM_IN_MM,
        Tags.DIM_VOLUME_Y_MM: VOLUME_PLANAR_DIM_IN_MM,
        Tags.WAVELENGTHS: [num for num in range(450, 1599, 10)], #[1400]
        Tags.DO_FILE_COMPRESSION: True
    }
    
    settings = sp.Settings(general_settings)
    
    settings.set_volume_creation_settings({
        Tags.SIMULATE_DEFORMED_LAYERS: True,
        Tags.STRUCTURES: simulation_structure
    })
    settings.set_optical_settings({
        Tags.OPTICAL_MODEL_NUMBER_PHOTONS: 5e7,
        Tags.OPTICAL_MODEL_BINARY_PATH: path_manager.get_mcx_binary_path(),
        Tags.COMPUTE_DIFFUSE_REFLECTANCE: SAVE_REFLECTANCE,
        Tags.COMPUTE_PHOTON_DIRECTION_AT_EXIT: SAVE_PHOTON_DIRECTION
        
    })
    settings["noise_model_1"] = {
        Tags.NOISE_MEAN: 1.0,
        Tags.NOISE_STD: 0.1,
        
        Tags.NOISE_MODE: Tags.NOISE_MODE_MULTIPLICATIVE,
        Tags.DATA_FIELD: Tags.DATA_FIELD_INITIAL_PRESSURE,
        Tags.NOISE_NON_NEGATIVITY_CONSTRAINT: True
    }
    
    pipeline = [
        sp.ModelBasedAdapter(settings),
        #
        sp.MCXReflectanceAdapter(settings)
    ]

    device = sp.PencilBeamIlluminationGeometry(device_position_mm=np.asarray([VOLUME_TRANSDUCER_DIM_IN_MM/2, VOLUME_PLANAR_DIM_IN_MM/2, 0]))
    
    
    sp.simulate(pipeline, settings, device)
    
    if Tags.WAVELENGTH in settings:
        WAVELENGTH = settings[Tags.WAVELENGTH]
    else:
        WAVELENGTH = 700
    
    if VISUALIZE:
        sp.visualise_data(path_to_hdf5_file=path_manager.get_hdf5_file_save_path() + "/" +  VOL_NAME + ".hdf5",
                          wavelength=WAVELENGTH,
                          show_initial_pressure=False,
                          show_absorption=True,
                          show_fluence=True,
                          show_diffuse_reflectance=True,
                          show_segmentation_map=True,
                          log_scale=False)
    return

In [ ]:
VOLUME_NAME_LIST = ["fat", "muscle", "Agar", "PDMS", "copolymer"]

structures_list = [subcutaneous_fat_tissue_structure(), muscle_tissue_structure(), agar_tissue_structure(), 
                   PDMS_tissue_structure(), copolymer_tissue_structure()]

for i, a in enumerate(structures_list):
    run_simulation(structures_list[i], VOLUME_NAME_LIST[i])